# 第 8 章 营销归因与预算优化

本章商业问题：**广告花费之后，哪些渠道真正带来可持续的经营回报？**

本章所有代码默认优先读取真实 ETL 接口 `http://192.168.31.47:38173/api/etl`。如果接口暂时不可用，会自动回退到本地 SQLite 后备数据，保证课堂可以继续进行。

## 0. 先建立商业问题意识

在商业课里，代码不是第一步。第一步是判断：这个问题影响收入、成本、用户体验、库存风险，还是营销效率？只有先说清楚商业目标，后面的数据选择和模型选择才不会跑偏。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if COURSE_ROOT.name in ["notebooks", "student_notebooks", "teacher_notebooks"]:
    COURSE_ROOT = COURSE_ROOT.parent
elif not (COURSE_ROOT / "course_utils").exists():
    COURSE_ROOT = Path("..").resolve()

sys.path.insert(0, str(COURSE_ROOT))

from course_utils.data_loader import (
    API_BASE, load_table, get_metrics, get_quality_report,
    get_table_catalog, get_schema, paid_orders, api_status, query_table
)
from course_utils.business import money, pct, section

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

print("课程目录:", COURSE_ROOT)
print("ETL API:", API_BASE)
print("API 状态:", api_status())

## 1. 查看 ETL 数据资产

下面先查看 ETL 接口暴露了哪些表。请注意 `dim_` 开头的是维度表，通常描述对象；`fact_` 开头的是事实表，通常记录业务事件。

In [ ]:
catalog = get_table_catalog()
tables = catalog["tables"]
print("可用表数量:", catalog.get("total", len(tables)))
for t in tables[:12]:
    print(t["tableName"], t.get("recordCount"), t.get("type"), t.get("description", ""))

In [ ]:
ads = load_table("fact_ads_spend", limit=100000)
channel = ads.groupby("channel").agg(spend=("spend_amount", "sum"), impressions=("impressions", "sum"), clicks=("clicks", "sum"), conversions=("conversions", "sum")).reset_index()
channel["ctr"] = channel["clicks"] / channel["impressions"]
channel["cvr"] = channel["conversions"] / channel["clicks"]
channel["cpa"] = channel["spend"] / channel["conversions"]
channel["estimated_revenue"] = channel["conversions"] * 120
channel["roas"] = channel["estimated_revenue"] / channel["spend"]
channel.sort_values("roas", ascending=False)

## 商业解释

点击率高说明广告吸引人，转化率高说明流量质量好，CPA 低说明获客便宜，ROAS 高说明花费能换回收入。但管理层最终还要问：这些转化是不是增量？是否抢了自然成交？

In [ ]:
assert channel["spend"].sum() > 0
print("第 08 章验证通过")

## 学生作业

请补充 150 到 300 字商业解读，至少引用两个数据结果，并给出一个明确运营动作。